# TurAIsta Model Evaluation: MMS and NLLB-200

This notebook runs the three models selected in the dataset audit and measures them on public benchmarks:

1. Speech recognition (MMS) on FLEURS test audio for Filipino and Cebuano, scored by word error rate.
2. Translation (NLLB-200) on FLORES+ from each guest language into each host language, scored by chrF++.
3. Text-to-speech (MMS) for a few farm-tour phrases in Tagalog, Cebuano, and Ilocano.

Benchmark test sets are used for evaluation only. Nothing in this notebook trains on them.

## Setup Checklist

1. Session options: set Accelerator to GPU T4 x1 and Internet to On.
2. Add-ons, Secrets: the `HF_TOKEN` secret from the audit notebook must be attached.
3. Run All. Expected runtime is 30 to 60 minutes.

## 1. Install and Configure

In [1]:
!pip install -q jiwer sacrebleu soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.8 MB/s eta 0:00:00


In [2]:
import os, io, csv, json, tarfile, time
from pathlib import Path
import numpy as np, soundfile as sf, torch
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
assert DEVICE == "cuda", "Turn on the GPU in Session options before running."

OUT = Path("/kaggle/working/eval_out"); OUT.mkdir(exist_ok=True)
N_ASR = 100     # FLEURS test utterances per language
N_MT = 100      # FLORES+ devtest sentences per direction
RESULTS = {}
def save():
    (OUT / "results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2))

Device: cuda


## 2. Speech Recognition: MMS on FLEURS

Audio and transcripts are read directly from the FLEURS repository files, since the dataset's loading script no longer runs on current versions of `datasets`. Word error rate (WER) is computed after lowercasing and removing punctuation on both sides.

In [3]:
import jiwer, re
from transformers import AutoProcessor, Wav2Vec2ForCTC

def norm(t):
    t = re.sub(r"[^\w\s'-]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()

def fleurs_test(code, n):
    tsv = hf_hub_download("google/fleurs", f"data/{code}/test.tsv", repo_type="dataset")
    rows = list(csv.reader(open(tsv, encoding="utf-8"), delimiter="\t", quoting=csv.QUOTE_NONE))
    seen, picked = set(), []
    for r in rows:                      # one utterance per sentence id
        if r[0] not in seen:
            seen.add(r[0]); picked.append(r)
        if len(picked) == n: break
    tgz = hf_hub_download("google/fleurs", f"data/{code}/audio/test.tar.gz", repo_type="dataset")
    want = {r[1]: r for r in picked}; audio = {}
    with tarfile.open(tgz) as t:
        for m in t:
            name = os.path.basename(m.name)
            if name in want:
                wav, sr = sf.read(io.BytesIO(t.extractfile(m).read()))
                audio[name] = (wav.astype(np.float32), sr)
    return [(audio[r[1]], r[3]) for r in picked if r[1] in audio]

processor = AutoProcessor.from_pretrained("facebook/mms-1b-all")
asr = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all").to(DEVICE).eval()

RESULTS["asr"] = {}
for lang, code, iso in [("tl", "fil_ph", "tgl"), ("ceb", "ceb_ph", "ceb")]:
    t0 = time.time()
    processor.tokenizer.set_target_lang(iso); asr.load_adapter(iso)
    data = fleurs_test(code, N_ASR); refs, hyps = [], []
    for (wav, sr), ref in data:
        assert sr == 16000
        x = processor(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            ids = asr(**x).logits.argmax(-1)[0]
        hyps.append(norm(processor.decode(ids))); refs.append(norm(ref))
    wer = jiwer.wer(refs, hyps)
    RESULTS["asr"][lang] = {"wer": round(wer, 4), "n": len(refs),
                            "examples": [{"ref": r, "hyp": h} for r, h in list(zip(refs, hyps))[:5]],
                            "seconds": round(time.time() - t0)}
    print(f"{lang}: WER {wer:.1%} on {len(refs)} utterances")
save()
del asr; torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

adapter.ceb.safetensors: downloading bytes:           |  0.00B            

test.tsv:   0%|          | 0.00/346k [00:00<?, ?B/s]

data/ceb_ph/audio/test.tar.gz: reconstructing file:   0%|          |  0.00B /  409MB            

data/ceb_ph/audio/test.tar.gz: downloading bytes:           |  0.00B            

ceb: WER 14.4% on 100 utterances


## 3. Translation: NLLB-200 on FLORES+

Each guest language (en, ko, ja, zh) is translated into each host language (tl, ceb, ilo), and the host languages are translated back into English. Sentences are aligned across languages by their FLORES `id`. The Tagalog config name is looked up rather than assumed, because the audit found that `tgl_Latn` is not listed.

In [4]:
import sacrebleu
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FLORES = "openlanguagedata/flores_plus"
configs = get_dataset_config_names(FLORES, token=HF_TOKEN)
tl_cfg = next((c for c in configs if c.startswith(("tgl_", "fil_"))), None)
print("Tagalog config:", tl_cfg)
FCFG = {"en": "eng_Latn", "ko": "kor_Hang", "ja": "jpn_Jpan", "zh": "cmn_Hans",
        "ceb": "ceb_Latn", "ilo": "ilo_Latn", "tl": tl_cfg}
NLLB = {"en": "eng_Latn", "ko": "kor_Hang", "ja": "jpn_Jpan", "zh": "zho_Hans",
        "tl": "tgl_Latn", "ceb": "ceb_Latn", "ilo": "ilo_Latn"}

# devtest is used here for EVALUATION ONLY
text = {}
for lang, cfg in FCFG.items():
    if cfg is None:
        print(f"skip {lang}: no config"); continue
    ds = load_dataset(FLORES, cfg, split="devtest", token=HF_TOKEN)
    text[lang] = {r["id"]: r["text"] for r in ds}
ids = sorted(set.intersection(*[set(v) for v in text.values()]))[:N_MT]

tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
mt = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M",
                                           torch_dtype=torch.float16).to(DEVICE).eval()

def translate(sents, src, tgt, bs=16):
    tok.src_lang = NLLB[src]; out = []
    for i in range(0, len(sents), bs):
        x = tok(sents[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            y = mt.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB[tgt]),
                            max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(y, skip_special_tokens=True)
    return out

RESULTS["mt"] = {}
pairs = [(g, h) for g in ["en", "ko", "ja", "zh"] for h in ["tl", "ceb", "ilo"]] + \
        [(h, "en") for h in ["tl", "ceb", "ilo"]]
for src, tgt in pairs:
    if src not in text or tgt not in text: continue
    srcs = [text[src][i] for i in ids]; refs = [text[tgt][i] for i in ids]
    hyps = translate(srcs, src, tgt)
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    RESULTS["mt"][f"{src}->{tgt}"] = {"chrf++": round(chrf, 1), "n": len(ids),
                                      "example": {"src": srcs[0], "hyp": hyps[0], "ref": refs[0]}}
    print(f"{src}->{tgt}: chrF++ {chrf:.1f}")
save()

README.md:   0%|          | 0.00/73.7k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

Tagalog config: fil_Latn


Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

...

tl->en: chrF++ 65.6


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

ceb->en: chrF++ 64.7


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

ilo->en: chrF++ 60.1


## 4. Farm-Tour Phrase Check

FLORES is news text. These short phrases are closer to what TurAIsta handles. There are no reference translations, so the outputs are for native-speaker review rather than scoring.

In [5]:
PHRASES = ["How much is the farm tour per person?",
           "Can we book for four people on Saturday morning?",
           "How do we get to your farm from the bus terminal?",
           "My son is allergic to peanuts. Is the food safe for him?",
           "Is the trail safe if it rains?"]
RESULTS["phrases"] = {}
for h in ["tl", "ceb", "ilo"]:
    RESULTS["phrases"][h] = dict(zip(PHRASES, translate(PHRASES, "en", h)))
    print(f"\n== en -> {h} (needs native-speaker review)")
    for s, t in RESULTS["phrases"][h].items():
        print(f"  {s}\n    {t}")
save()
del mt; torch.cuda.empty_cache()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



== en -> tl (needs native-speaker review)
  How much is the farm tour per person?
    Magkano ang gastos ng farm tour sa bawat tao?
  Can we book for four people on Saturday morning?
    Maaari ba tayong mag-book para sa apat na tao sa umaga ng Sabado?
  How do we get to your farm from the bus terminal?
    Paano tayo pupunta sa inyong farm mula sa terminal ng bus?
  My son is allergic to peanuts. Is the food safe for him?
    Ang aking anak ay allergic sa mga buto. ligtas ba ang pagkain para sa kanya?
  Is the trail safe if it rains?
    ligtas ba ang landas kung umuulan?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



== en -> ceb (needs native-speaker review)
  How much is the farm tour per person?
    Pila ang kantidad sa paglibot sa uma matag tawo?
  Can we book for four people on Saturday morning?
    Mahimo ba kitang mag-book alang sa upat ka tawo sa Sabado sa buntag?
  How do we get to your farm from the bus terminal?
    Sa unsang paagi kita moabot sa imong uma gikan sa terminal sa bus?
  My son is allergic to peanuts. Is the food safe for him?
    Ang akong anak nga lalaki alergik sa mga mansanas. Segurado ba ang pagkaon alang kaniya?
  Is the trail safe if it rains?
    Malipayon ba ang dalan kon mouran?

== en -> ilo (needs native-speaker review)
  How much is the farm tour per person?
    Mano ti magastos ti tour iti talon iti kada tao?
  Can we book for four people on Saturday morning?
    Mabalin kadi nga agbiroktayo iti uppat a tattao iti agsapa ti Sabado?
  How do we get to your farm from the bus terminal?
    Kasano ti isasangpetmi iti awaymo manipud iti terminal ti bus?
  My son is

## 5. Text-to-Speech: MMS

Each host language speaks one host-side phrase. The phrases are drafts and need native-speaker review. Play the audio below each language, and judge whether a host would understand it.

In [6]:
from transformers import VitsModel, AutoTokenizer as TTSTok
from IPython.display import Audio, display

SAY = {"tl": "Magandang umaga po! Handa na po ang almusal.",
       "ceb": "Maayong buntag! Andam na ang pamahaw.",
       "ilo": "Naimbag a bigat! Nakasagana metten ti pammigat."}
RESULTS["tts"] = {}
for lang, iso in [("tl", "tgl"), ("ceb", "ceb"), ("ilo", "ilo")]:
    m = VitsModel.from_pretrained(f"facebook/mms-tts-{iso}").to(DEVICE).eval()
    t = TTSTok.from_pretrained(f"facebook/mms-tts-{iso}")
    with torch.no_grad():
        wav = m(**t(SAY[lang], return_tensors="pt").to(DEVICE)).waveform[0].cpu().numpy()
    path = OUT / f"tts_{lang}.wav"; sf.write(path, wav, m.config.sampling_rate)
    RESULTS["tts"][lang] = {"text": SAY[lang], "file": path.name,
                            "seconds": round(len(wav) / m.config.sampling_rate, 2)}
    print(lang, SAY[lang]); display(Audio(str(path)))
save()

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

ilo Naimbag a bigat! Nakasagana metten ti pammigat.


## 6. Summary and Package

In [7]:
print("Speech recognition (WER, lower is better):")
for k, v in RESULTS.get("asr", {}).items(): print(f"  {k}: {v['wer']:.1%}")
print("Translation (chrF++, higher is better):")
for k, v in RESULTS.get("mt", {}).items(): print(f"  {k}: {v['chrf++']}")
import shutil
shutil.make_archive("/kaggle/working/eval_out", "zip", OUT)
from IPython.display import FileLink
os.chdir("/kaggle/working"); FileLink("eval_out.zip")

Speech recognition (WER, lower is better):
  tl: 13.1%
  ceb: 14.4%
Translation (chrF++, higher is better):
  en->tl: 57.5
  en->ceb: 55.8
  en->ilo: 52.4
  ko->tl: 42.7
  ko->ceb: 41.3
  ko->ilo: 40.1
  ja->tl: 42.3
  ja->ceb: 42.0
  ja->ilo: 39.4
  zh->tl: 42.8
  zh->ceb: 43.8
  zh->ilo: 40.8
  tl->en: 65.6
  ceb->en: 64.7
  ilo->en: 60.1


/kaggle/working/eval_out.zip